In [1]:
## A40 ##

# STEP 0: Downgrade NumPy to prevent compatibility crashes
!pip install numpy<2.0 -q --upgrade > /dev/null 2>&1

# STEP 1: Ensure clean install & safe pip version
!pip install pip==23.3.1 --upgrade -q > /dev/null 2>&1
!pip install wheel setuptools --upgrade -q > /dev/null 2>&1

# STEP 2: Install PyTorch 2.1.0 + CUDA 11.8
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 -q > /dev/null 2>&1

# STEP 3: Install all required packages silently
!pip install -q \
  transformers==4.44.0 \
  peft==0.12.0 \
  accelerate==0.33.0 \
  datasets==2.17.1 \
  scipy==1.12.0 \
  trl==0.9.6 \
  hf_transfer==0.1.5 \
  huggingface_hub==0.24.0 \
  wandb==0.16.3 \
  matplotlib > /dev/null 2>&1

# STEP 4: Install flash-attn (compiled wheels)
!pip install flash-attn==2.5.5 --extra-index-url https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.5/ -q > /dev/null 2>&1

# STEP 5: Check Transformers env — should now work
!transformers-cli env

/bin/bash: line 1: 2.0: No such file or directory

Copy-and-paste the text below in your GitHub issue and FILL OUT the two last points.

- `transformers` version: 4.44.0
- Platform: Linux-6.8.0-65-generic-x86_64-with-glibc2.35
- Python version: 3.11.10
- Huggingface_hub version: 0.24.0
- Safetensors version: 0.7.0
- Accelerate version: 0.33.0
- Accelerate config: 	not found
- PyTorch version (GPU?): 2.1.0+cu118 (True)
- Tensorflow version (GPU?): not installed (NA)
- Flax version (CPU?/GPU?/TPU?): not installed (NA)
- Jax version: not installed
- JaxLib version: not installed
- Using distributed or parallel set-up in script?: <fill in>
- Using GPU in script?: <fill in>
- GPU type: NVIDIA A40



In [2]:
def generate_device_map(num_layers: int, num_gpus: int) -> dict:
    device_map = {}
    layers_per_gpu = num_layers // num_gpus
    remainder = num_layers % num_gpus

    layer_id = 0
    for gpu in range(num_gpus):
        this_gpu_layer_count = layers_per_gpu + (1 if gpu < remainder else 0)
        for _ in range(this_gpu_layer_count):
            device_map[f"model.layers.{layer_id}"] = gpu
            layer_id += 1

    device_map["model.embed_tokens"] = 0
    device_map["model.norm"] = num_gpus - 1
    device_map["lm_head"] = num_gpus - 1

    return device_map


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

old_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
adapter_id = 'agi123/DeepSeek-R1-Distill-Llama-8B-orpo-orpo_health_insurance_deepseek-epochs-5adapters'

# Load base model
model_old = AutoModelForCausalLM.from_pretrained(
    old_id,
    torch_dtype=torch.bfloat16,
    device_map=generate_device_map(32,1),
    trust_remote_code=True
)


eval_model = PeftModel.from_pretrained(model_old, adapter_id)
#eval_model = eval_model.to("cuda")
eval_model.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(old_id,use_fast=True,trust_remote_code=True)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.67G [00:00<?, ?B/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/7.39G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

In [4]:
eval_model.config.use_cache = True

In [9]:
# import re

# def evaluate(question, model, tokenizer):
#     messages = [
#         {"role": "user", "content": question}
#     ]
    
#     formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
#     # Generate response
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=500,
#         temperature=0.6,
#         top_p=0.95,
#         do_sample=True,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )
    
#     # Decode the response - try WITHOUT skip_special_tokens first to see raw output
#     raw_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
#     print("=" * 50)
#     print("RAW OUTPUT:")
#     print(raw_text)
#     print("=" * 50)
    
#     # Now decode with skip_special_tokens
#     generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     print("AFTER skip_special_tokens=True:")
#     print(generated_text)
#     print("=" * 50)
    
#     # Remove the prompt/question part
#     if question in generated_text:
#         response = generated_text.split(question)[-1].strip()
#     else:
#         response = generated_text
    
#     print("AFTER removing question:")
#     print(response)
#     print("=" * 50)
    
#     return response

In [5]:
import re

def evaluate(question, model, tokenizer):
    messages = [
        {"role": "user", "content": question}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.6,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    # Decode the response
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Remove the prompt/question part
    if question in generated_text:
        response = generated_text.split(question)[-1].strip()
    else:
        response = generated_text
    
    # Remove DeepSeek tags (with regular pipe characters)
    response = response.replace('<|Assistant|>', '')
    response = response.replace('<|User|>', '')
    response = response.replace('<|begin_of_sentence|>', '')
    response = response.replace('<|end_of_sentence|>', '')
    
    # Remove <think> tag (empty or with content)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL)
    response = response.replace('<think>', '')  # Remove standalone <think> tag
    response = response.replace('</think>', '')  # Remove standalone </think> tag
    response = response.replace('<｜Assistant｜>', '')
    
    # Clean up whitespace
    response = response.strip()
    
    return response


In [6]:
#Earmarked Question - 1

q = 'According to HDFC ERGO Optima Restore, What if I use up all my sum insured—do I have to pay out-of-pocket after that?'

print(evaluate(q, eval_model, tokenizer))

If you use up all your sum insured, you do not have to pay out-of-pocket for the admissible claim.


In [7]:
#Earmarked Question - 2

q = 'According to HDFC ERGO Optima Restore, Will the restored sum insured carry forward if unused?'

evaluate(q, eval_model, tokenizer)

'According to HDFC ERGO Optima Restore, the restored sum insured is not carried forward if unused.'

In [8]:
#Earmarked Question - 2 - follow Up

q = 'But the policy document says "Nope—its a use-it-or-lose-it deal and resets every policy year.", what do you say of this?'

evaluate(q, eval_model, tokenizer)

'The policy document states that the policy is a "No-Notice period" where the insured must notify the insurer within 15 days of the policy\'s commencement or renewal to make any changes. Failure to do so will result in the policy lapsing.'

In [9]:
q = 'According to HDFC ERGO Optima Restore, So if I get admitted, do I have to first pay and then beg for money later?'

evaluate(q, eval_model, tokenizer)

'No, according to HDFC ERGO Optima Restore, if you get admitted, you do not have to pay and then seek reimbursement. The company will pay for the admissible claim directly to the hospital and provider, and you will receive the insured amount directly.'

# Overfitting test

In [10]:
#1# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, I will still need some time for rest and recovery right, will this policy cover my expenses even then?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, After discharging from the hospital, I will still need some time for rest and recovery right, will this policy cover my expenses even then?assistant

Yes, the HDFC ERGO Optima Restore policy will cover your expenses even after you have been discharged from the hospital. The policy covers expenses for consultations, investigations, and medicines incurred during the 'Post-Hospitalisation' period, which is up to 180 days after discharge from the hospital. Additionally, it also covers expenses for consultations, investigations, and medicines incurred during the 'Pre-Hospitalisation' period, which is up to 60 days before the hospitalisation.


In [11]:
# tweaked question
# Accuracy Rating = 0.5
# Verdict: Overfitting  (Hallucination)

q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, Will I get the coverage for the recovery period were I can not go to work for 2 to 3 months?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, After discharging from the hospital, Will I get the coverage for the recovery period were I can not go to work for 2 to 3 months?assistant

Yes, you will get the coverage for the recovery period where you cannot go to work for 2 to 3 months after discharging from the hospital under the HDFC ERGO Optima Restore policy. This is known as the 'Rehabilitation period'. It is essential to ensure that the break in policy is not more than 90 days, and the payment for the current policy must be made in full and on time to avoid any policy break.


In [41]:
#2# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, Does the policy pay for infertility treatment or IVF?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, Does the policy pay for infertility treatment or IVF?assistant

No, the HDFC ERGO Optima Restore policy does not cover expenses incurred on infertility treatment, including IVF. This is specified in List I of items for which the policy does not apply.


In [42]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = 'Is infertility treatment or IVF paid for by the policy?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

Is infertility treatment or IVF paid for by the policy?assistant

No, infertility treatment or IVF is not covered under the HDFC ERGO Optima Restore policy. Any expenses incurred on artificial insemination, gestational surrogacy, and reverse vasectomy are also not covered.


In [43]:
#3# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, If I get injured doing adventure sports, will they pay?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, If I get injured doing adventure sports, will they pay?assistant

No, HDFC ERGO Optima Restore does not cover expenses related to any treatment necessitated due to participation as a professional in any adventure or hazardous activity or any attempt to commit suicide. This is excluded under Section 1.i of the policy.


In [44]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = 'If I got injured doing bunjee jumping, will this policy cover my medical bills?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

If I got injured doing bunjee jumping, will this policy cover my medical bills?assistant

Bungee jumping is considered an adventure or extreme sport, and most health insurance policies do not cover medical expenses incurred while participating in such activities. However, the specific terms and conditions of your policy should be reviewed to determine the extent of coverage. It is essential to check the policy wording for any exclusions related to adventure or hazardous sports. If the injury is due to an accident and not related to the adventure or extreme sport, the policy may cover the medical expenses. It is crucial to provide the insurer with detailed information about the circumstances surrounding the injury to determine the validity of the claim.


In [45]:
#4# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?assistant

Yes, if you get diagnosed with a critical illness under the HDFC ERGO Optima Restore policy, you can get a second opinion from another doctor, which can be covered by insurance. The policy allows for consultation fees to be covered for the second opinion.


In [18]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = "If I'm diagnosed with a critical illness, will HDFC ERGO Optima Restore let me get a second opinion and cover it too?"

evaluate(q, eval_model, tokenizer)


"If I'm diagnosed with a critical illness, I have the right to get a second opinion on the diagnosis and treatment plan from a specialist medical practitioner registered with the IRDAI. The insurer will pay for the second opinion, and the diagnosis and treatment will also be covered."

In [ ]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = "I'm diagnosed with a critical illness, This policy says go get a second opi"

evaluate(q, eval_model, tokenizer)


# Update accuracyCheck.xlsx with the adapter answers.

In [10]:
!pip install openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 8.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [11]:
import pandas as pd
from tqdm import tqdm

excel_path = "accuracyCheck.xlsx"
output_path = "accuracyCheck_deepseek_8b_ep5_bs2_evaluated.xlsx"

df = pd.read_excel(excel_path)

for idx, row in tqdm(df.iterrows(), total=len(df)):
    if pd.notna(row['question']):
        response = ""
        for chunk in evaluate(row['question'], eval_model, tokenizer):
            response += chunk
        df.at[idx, 'adapter_model_answer'] = response.strip()
        
        if (idx + 1) % 10 == 0:
            df.to_excel(output_path, index=False)

df.to_excel(output_path, index=False)
print("Done!")

  0%|          | 0/30 [00:00<?, ?it/s]/tmp/ipykernel_2184/3119774968.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'HDFC ERGO Optima Restore is a health insurance policy that offers a wide range of sum insured options, including 3.5 Lacs, 5 Lacs, 10 Lacs, 15 Lacs, 20 Lacs, 25 Lacs, 50 Lacs, 100 Lacs, 150 Lacs, 200 Lacs, 300 Lacs, 500 Lacs, 1000 Lacs, and 1.5 Lacs. It provides coverage for in-patient hospitalization expenses, day care procedures, pre and post-hospitalization expenses, emergency air ambulance, and domiciliary treatment. The policy also includes optional covers such as preventive health checkup, E-opinion, and AYUSH treatment. It is underwritten by HDFC ERGO General Insurance Company Limited.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'adapter_model_answer'] = response.strip()
100%|██████████| 30/30 [01:13<00:00,  2.44s/it]

Done!


# Calculate Accuracy & Push to W&B

In [13]:
import pandas as pd
import wandb
import json
from datetime import datetime

# ----- CONFIGURATION -----
EXCEL_PATH = "accuracyCheck_deepseek_8b_ep5_bs2_scored.xlsx"
WANDB_ENTITY = "radhakrishna5a99-innomatics-research-labs"
WANDB_PROJECT = "orpo_finetune_DeepSeek-R1-Distill-Llama-8B"
ARTIFACT_NAME = "evaluation-orpo_run_DeepSeek-R1-Distill-Llama-8B_orpo_health_insurance_deepseek_config_and_results"
ARTIFACT_VERSION = "v0"

# ----- LOAD & CALCULATE -----
df = pd.read_excel(EXCEL_PATH)
scores = df['score'].tolist()

total_questions = len(scores)
score_1_count = scores.count(1)
score_05_count = scores.count(0.5)
score_0_count = scores.count(0)

# Accuracy Calculations
accuracy_percentage = (sum(scores) / total_questions) * 100
strict_accuracy = (score_1_count / total_questions) * 100
lenient_accuracy = ((score_1_count + score_05_count) / total_questions) * 100

print("="*50)
print("ACCURACY METRICS")
print("="*50)
print(f"Total Questions: {total_questions}")
print(f"Correct (1.0):   {score_1_count} ({score_1_count/total_questions*100:.1f}%)")
print(f"Partial (0.5):   {score_05_count} ({score_05_count/total_questions*100:.1f}%)")
print(f"Incorrect (0.0): {score_0_count} ({score_0_count/total_questions*100:.1f}%)")
print(f"\nAccuracy Percentage: {accuracy_percentage:.2f}%")
print(f"Strict Accuracy:     {strict_accuracy:.2f}%")
print(f"Lenient Accuracy:    {lenient_accuracy:.2f}%")
print("="*50)

# ----- PUSH TO WANDB -----
# Initialize run
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    job_type="evaluation",
    name=f"accuracy_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

# Prepare metrics
metrics = {
    "total_questions": total_questions,
    "score_1_count": score_1_count,
    "score_0.5_count": score_05_count,
    "score_0_count": score_0_count,
    "accuracy_percentage": round(accuracy_percentage, 2),
    "strict_accuracy": round(strict_accuracy, 2),
    "lenient_accuracy": round(lenient_accuracy, 2),
}

# Log metrics
wandb.log(metrics)
wandb.summary.update(metrics)

# Create evaluation artifact
eval_artifact = wandb.Artifact(
    name=f"evaluation-{ARTIFACT_NAME}",
    type="evaluation",
    description=f"Accuracy evaluation for {ARTIFACT_NAME}",
    metadata=metrics
)

# Add Excel file
eval_artifact.add_file(EXCEL_PATH)

# Save and add metrics JSON
with open("accuracy_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)
eval_artifact.add_file("accuracy_metrics.json")

# Log artifact
run.log_artifact(eval_artifact)

# Link to original model artifact
try:
    model_artifact = run.use_artifact(
        f"{WANDB_ENTITY}/{WANDB_PROJECT}/{ARTIFACT_NAME}:{ARTIFACT_VERSION}"
    )
    print(f"\nLinked to model artifact: {model_artifact.name}")
except Exception as e:
    print(f"\nNote: Could not link to model artifact: {e}")

# Log as table for visualization
wandb.log({"evaluation_table": wandb.Table(dataframe=df)})

# Finish
run.finish()

print(f"\nSuccessfully pushed to W&B!")
print(f"Project: {WANDB_ENTITY}/{WANDB_PROJECT}")
print(f"Artifact: evaluation-{ARTIFACT_NAME}")

ACCURACY METRICS
Total Questions: 30
Correct (1.0):   13 (43.3%)
Partial (0.5):   10 (33.3%)
Incorrect (0.0): 7 (23.3%)

Accuracy Percentage: 60.00%
Strict Accuracy:     43.33%
Lenient Accuracy:    76.67%


/usr/local/lib/python3.11/dist-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
wandb: Currently logged in as: radhakrishna5a99 (radhakrishna5a99-innomatics-research-labs). Use `wandb login --relogin` to force relogin



Linked to model artifact: evaluation-orpo_run_DeepSeek-R1-Distill-Llama-8B_orpo_health_insurance_deepseek_config_and_results:v0


accuracy_percentage,▁
lenient_accuracy,▁
score_0.5_count,▁
score_0_count,▁
score_1_count,▁
strict_accuracy,▁
total_questions,▁
accuracy_percentage,60.0
lenient_accuracy,76.67
score_0.5_count,10
score_0_count,7



Successfully pushed to W&B!
Project: radhakrishna5a99-innomatics-research-labs/orpo_finetune_DeepSeek-R1-Distill-Llama-8B
Artifact: evaluation-evaluation-orpo_run_DeepSeek-R1-Distill-Llama-8B_orpo_health_insurance_deepseek_config_and_results
